# Week 2 Day 3 — LangGraph: Stateful, Multi-Step & Cyclical Agent Workflows

A research assistant with planning, retrieval, generation, critique, self-correction, human approval, persistence, and debugging.

## Learning objectives
- Understand `StateGraph`, nodes, edges, conditional edges, and shared state.
- Build a linear graph and extend it with a self-correction cycle.
- Pause before a risky action and resume after human approval or rejection.
- Persist state with `MemorySaver` and inspect execution history.


## 1. Core graph concepts

- **StateGraph:** the graph builder that defines the workflow and its state schema.
- **Node:** a Python function that receives the current state and returns state updates.
- **Edge:** a fixed transition from one node to another.
- **Conditional edge:** a routing function that chooses the next node based on state.
- **Shared State:** the structured data passed between nodes. It is the graph's memory for the current run.

### State design

This project uses a `TypedDict` containing the question, plan, retrieved evidence, draft, critique, quality score, retry counter, approval decision, and final answer.


## 2. Graph diagram

```mermaid
flowchart TD
    A([Start]) --> B[Plan]
    B --> C[Retrieve]
    C --> D[Generate]
    D --> E[Critique]
    E -->|score < 0.8 and retries < 2| D
    E -->|acceptable or retry limit reached| F[Human approval]
    F -->|approved| G[Format]
    F -->|rejected| H[Revise request]
    H --> D
    G --> I([End])
```


In [1]:
!pip install -q --break-system-packages langgraph langchain-core

In [ ]:
from typing import TypedDict, List
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command


In [3]:
class ResearchState(TypedDict, total=False):
    question: str
    plan: List[str]
    evidence: List[str]
    draft: str
    critique: str
    quality_score: float
    retries: int
    max_retries: int
    approval: str
    final_answer: str
    history: List[str]


## 3. Day 2-style tools

For a self-contained notebook, retrieval uses a small local knowledge base. The same node can later call a real search tool, vector database, or API.

In [4]:
KNOWLEDGE_BASE = {
    'langgraph': 'LangGraph models workflows as graphs of stateful nodes and transitions.',
    'state': 'Shared state carries information between nodes and can be checkpointed.',
    'human': 'Human approval is useful before irreversible, expensive, or externally visible actions.'
}

def local_search(query: str) -> list[str]:
    words = query.lower().split()
    matches = [text for key, text in KNOWLEDGE_BASE.items()
               if key in words or key in query.lower()]
    return matches or list(KNOWLEDGE_BASE.values())[:2]


## 4. Linear graph: plan → retrieve → generate → format

In [5]:
def plan_node(state: ResearchState):
    question = state['question']
    return {'plan': [f'Identify the main concepts in: {question}',
                     'Retrieve supporting information',
                     'Write a concise answer'],
            'history': state.get('history', []) + ['plan']}

def retrieve_node(state: ResearchState):
    evidence = local_search(state['question'])
    return {'evidence': evidence,
            'history': state.get('history', []) + ['retrieve']}

def generate_node(state: ResearchState):
    evidence = ' '.join(state.get('evidence', []))
    feedback = state.get('critique') or state.get('approval')
    draft = (f"Question: {state['question']}\n\n"
             f"Answer: {evidence}"
             f"\n\nRevision pass: {state.get('retries', 0)}")
    if feedback:
        draft += f"\n\n(incorporating feedback: {feedback})"
    return {'draft': draft,
            'history': state.get('history', []) + ['generate']}

def format_node(state: ResearchState):
    return {'final_answer': state['draft'].strip(),
            'history': state.get('history', []) + ['format']}

linear = StateGraph(ResearchState)
linear.add_node('plan', plan_node)
linear.add_node('retrieve', retrieve_node)
linear.add_node('generate', generate_node)
linear.add_node('format', format_node)
linear.add_edge(START, 'plan')
linear.add_edge('plan', 'retrieve')
linear.add_edge('retrieve', 'generate')
linear.add_edge('generate', 'format')
linear.add_edge('format', END)
linear_graph = linear.compile()

result = linear_graph.invoke({'question': 'What is LangGraph?', 'retries': 0})
result

{'question': 'What is LangGraph?',
 'plan': ['Identify the main concepts in: What is LangGraph?',
  'Retrieve supporting information',
  'Write a concise answer'],
 'evidence': ['LangGraph models workflows as graphs of stateful nodes and transitions.'],
 'draft': 'Question: What is LangGraph?\n\nAnswer: LangGraph models workflows as graphs of stateful nodes and transitions.\n\nRevision pass: 0',
 'retries': 0,
 'final_answer': 'Question: What is LangGraph?\n\nAnswer: LangGraph models workflows as graphs of stateful nodes and transitions.\n\nRevision pass: 0',
 'history': ['plan', 'retrieve', 'generate', 'format']}

In [6]:
# Verify the state after every node
for snapshot in linear_graph.stream(
    {'question': 'Why is shared state useful?', 'retries': 0},
    stream_mode='values'
):
    print('STATE UPDATE:', snapshot)


STATE UPDATE: {'question': 'Why is shared state useful?', 'retries': 0}
STATE UPDATE: {'question': 'Why is shared state useful?', 'plan': ['Identify the main concepts in: Why is shared state useful?', 'Retrieve supporting information', 'Write a concise answer'], 'retries': 0, 'history': ['plan']}
STATE UPDATE: {'question': 'Why is shared state useful?', 'plan': ['Identify the main concepts in: Why is shared state useful?', 'Retrieve supporting information', 'Write a concise answer'], 'evidence': ['Shared state carries information between nodes and can be checkpointed.'], 'retries': 0, 'history': ['plan', 'retrieve']}
STATE UPDATE: {'question': 'Why is shared state useful?', 'plan': ['Identify the main concepts in: Why is shared state useful?', 'Retrieve supporting information', 'Write a concise answer'], 'evidence': ['Shared state carries information between nodes and can be checkpointed.'], 'draft': 'Question: Why is shared state useful?\n\nAnswer: Shared state carries information bet

## 5. Conditional edges and self-correction

The critique node evaluates the draft. If the score is below `0.8`, the graph returns to `generate`. The retry counter guarantees termination.

In [7]:
def critique_node(state: ResearchState):
    draft = state.get('draft', '')
    retries = state.get('retries', 0)
    # Demonstration rule: short drafts fail once, then pass after revision.
    score = 0.65 if retries == 0 else 0.92
    critique = 'Add more detail and improve completeness.' if score < 0.8 else 'Draft is acceptable.'
    print(f'Critique pass {retries + 1}: score={score}, {critique}')
    return {'quality_score': score,
            'critique': critique,
            'retries': retries + 1,
            'history': state.get('history', []) + ['critique']}

def route_after_critique(state: ResearchState):
    if state['quality_score'] < 0.8 and state['retries'] <= state.get('max_retries', 2):
        return 'generate'
    return 'approval'

def approval_node(state: ResearchState):
    decision = interrupt({
        'message': 'Approve releasing this answer?',
        'draft': state['draft'],
        'quality_score': state['quality_score']
    })
    return {'approval': decision,
            'history': state.get('history', []) + ['approval']}

def route_after_approval(state: ResearchState):
    return 'format' if state.get('approval') == 'approved' else 'revise'

def revise_node(state: ResearchState):
    return {'draft': state['draft'] + '\n\nRevision: The answer was reviewed and clarified.',
            'history': state.get('history', []) + ['revise']}


In [8]:
builder = StateGraph(ResearchState)
for name, fn in [('plan', plan_node), ('retrieve', retrieve_node),
                 ('generate', generate_node), ('critique', critique_node),
                 ('approval', approval_node), ('revise', revise_node),
                 ('format', format_node)]:
    builder.add_node(name, fn)

builder.add_edge(START, 'plan')
builder.add_edge('plan', 'retrieve')
builder.add_edge('retrieve', 'generate')
builder.add_edge('generate', 'critique')
builder.add_conditional_edges(
    'critique', route_after_critique, {'generate': 'generate', 'approval': 'approval'}
)
builder.add_conditional_edges(
    'approval', route_after_approval, {'format': 'format', 'revise': 'revise'}
)
builder.add_edge('revise', 'generate')
builder.add_edge('format', END)

memory = MemorySaver()
graph = builder.compile(checkpointer=memory)


### Human-in-the-loop execution

The graph pauses at `approval`. A real application would display the draft to a user. Here we simulate approval by resuming with `Command(resume='approved')`.

In [9]:
config = {'configurable': {'thread_id': 'research-demo-1'}}
initial = {
    'question': 'Explain why LangGraph uses shared state.',
    'retries': 0,
    'max_retries': 2,
    'history': []
}

paused = graph.invoke(initial, config)
print('Graph paused for approval.')
print(paused)

# Simulated human approval
resumed = graph.invoke(Command(resume='approved'), config)
print('Resumed result:')
print(resumed['final_answer'])


Critique pass 1: score=0.65, Add more detail and improve completeness.
Critique pass 2: score=0.92, Draft is acceptable.
Graph paused for approval.
{'question': 'Explain why LangGraph uses shared state.', 'plan': ['Identify the main concepts in: Explain why LangGraph uses shared state.', 'Retrieve supporting information', 'Write a concise answer'], 'evidence': ['LangGraph models workflows as graphs of stateful nodes and transitions.', 'Shared state carries information between nodes and can be checkpointed.'], 'draft': 'Question: Explain why LangGraph uses shared state.\n\nAnswer: LangGraph models workflows as graphs of stateful nodes and transitions. Shared state carries information between nodes and can be checkpointed.\n\nRevision pass: 1\n\n(incorporating feedback: Add more detail and improve completeness.)', 'critique': 'Draft is acceptable.', 'quality_score': 0.92, 'retries': 2, 'max_retries': 2, 'history': ['plan', 'retrieve', 'generate', 'critique', 'generate', 'critique'], '__i

### Rejection path

If the human returns anything other than `'approved'`, the conditional edge routes to `revise`, then back to `generate`. In a production system, the rejection could include feedback such as `'Add citations'` or `'Remove unsupported claims'`.

In [10]:
# Example rejection flow using a new thread
reject_config = {'configurable': {'thread_id': 'research-demo-rejected'}}
graph.invoke(initial, reject_config)
rejected = graph.invoke(Command(resume='rejected'), reject_config)
print('After rejection, routed to revise -> generate -> critique -> approval again:')
print('history:', rejected['history'])
print('draft now includes revision feedback:')
print(rejected['draft'])
print()

# Second human decision: approve this time, so the loop actually terminates
final = graph.invoke(Command(resume='approved'), reject_config)
print('Final answer after rejection + revision + approval:')
print(final['final_answer'])

Critique pass 1: score=0.65, Add more detail and improve completeness.
Critique pass 2: score=0.92, Draft is acceptable.
Critique pass 3: score=0.92, Draft is acceptable.
After rejection, routed to revise -> generate -> critique -> approval again:
history: ['plan', 'retrieve', 'generate', 'critique', 'generate', 'critique', 'approval', 'revise', 'generate', 'critique']
draft now includes revision feedback:
Question: Explain why LangGraph uses shared state.

Answer: LangGraph models workflows as graphs of stateful nodes and transitions. Shared state carries information between nodes and can be checkpointed.

Revision pass: 2

(incorporating feedback: Draft is acceptable.)

Final answer after rejection + revision + approval:
Question: Explain why LangGraph uses shared state.

Answer: LangGraph models workflows as graphs of stateful nodes and transitions. Shared state carries information between nodes and can be checkpointed.

Revision pass: 2

(incorporating feedback: Draft is acceptable

## 6. Persistence and debugging

`MemorySaver` stores checkpoints for the configured thread. Reusing the same `thread_id` allows the application to resume a paused execution. For durable production persistence, use a database-backed checkpointer such as PostgreSQL rather than process-local memory.


In [11]:
# Inspect checkpoints/history for one run
for i, snapshot in enumerate(graph.get_state_history(config)):
    print(f'--- Snapshot {i} ---')
    print('next:', snapshot.next)
    print('state:', snapshot.values)


--- Snapshot 0 ---
next: ()
state: {'question': 'Explain why LangGraph uses shared state.', 'plan': ['Identify the main concepts in: Explain why LangGraph uses shared state.', 'Retrieve supporting information', 'Write a concise answer'], 'evidence': ['LangGraph models workflows as graphs of stateful nodes and transitions.', 'Shared state carries information between nodes and can be checkpointed.'], 'draft': 'Question: Explain why LangGraph uses shared state.\n\nAnswer: LangGraph models workflows as graphs of stateful nodes and transitions. Shared state carries information between nodes and can be checkpointed.\n\nRevision pass: 1\n\n(incorporating feedback: Add more detail and improve completeness.)', 'critique': 'Draft is acceptable.', 'quality_score': 0.92, 'retries': 2, 'max_retries': 2, 'approval': 'approved', 'final_answer': 'Question: Explain why LangGraph uses shared state.\n\nAnswer: LangGraph models workflows as graphs of stateful nodes and transitions. Shared state carries in

## 7. Why the cycle is natural in LangGraph

A plain `AgentExecutor` generally hides the control loop inside an agent runtime, so custom retry rules, explicit state fields, approval pauses, and multiple branches become harder to inspect and control. LangGraph makes the loop a first-class conditional edge, while the state object records exactly why the graph revisited a node and how many retries remain.

## 8. Human-in-the-loop policy

**Require approval** before irreversible, expensive, legally sensitive, privacy-sensitive, or externally visible actions—for example, sending an email, deleting data, purchasing something, publishing content, or changing production infrastructure.

**Full autonomy** is reasonable for low-risk, reversible, well-bounded tasks such as summarizing documents, classifying messages, drafting code, or retrieving public information, provided that monitoring, access controls, and failure limits exist.

## 9. AgentExecutor vs LangGraph

| Use case | AgentExecutor | LangGraph |
|---|---|---|
| Quick tool-using agent | Excellent | Also possible, but more setup |
| Fixed or mostly linear tool loop | Simple | Useful when state visibility matters |
| Branching and cycles | Less explicit | First-class conditional edges |
| Human approval and pause/resume | More custom plumbing | Built-in interrupt/checkpoint model |
| Long-running workflows | Limited without extra design | Strong fit with persistence |
| Debugging and replay | Usually manual | State history and checkpoints |

**Rule of thumb:** reach for `AgentExecutor` for a small autonomous tool-calling assistant; choose LangGraph when the workflow is stateful, multi-step, cyclical, auditable, or needs human control.